<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/answer/Chapter10_lm_head_answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 10 — LM Head — hidden state → next-token logits**

> ✅ Answer notebook — full working code.

## Goal

After the blocks we have `(B, T, C)` hidden states. To predict the next token we need a score for
every vocabulary entry at every position: `(B, T, vocab_size)`. That's a final LayerNorm + a single
Linear (the **language-model head**). Softmax over the last dim turns logits into probabilities.

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────
# Pure PyTorch — no extra installs needed beyond torch (preinstalled on Colab).
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

In [ ]:
C, vocab_size = 32, 65

ln_f    = nn.LayerNorm(C)
lm_head = nn.Linear(C, vocab_size)

h = torch.randn(4, 8, C)               # final hidden states (B, T, C)
logits = lm_head(ln_f(h))              # (B, T, vocab_size)
print("logits:", logits.shape)         # (4, 8, 65)

probs = F.softmax(logits, dim=-1)
print("probs sum to 1:", probs[0, 0].sum().item())

## The loss

Training compares logits to the true next token with **cross-entropy**. PyTorch wants
`(B*T, vocab)` logits and `(B*T,)` targets, so we flatten.

In [ ]:
targets = torch.randint(0, vocab_size, (4, 8))      # pretend next-token labels
loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
print("loss:", loss.item())
print("random-guess loss ≈ ln(vocab) =", round(torch.log(torch.tensor(float(vocab_size))).item(), 3))

✅ Every piece exists. Next: assemble the full nanoGPT.